In [ ]:
# Install required dependencies
# %pip install pyspark==3.5.1


In [4]:
from config import *

In [ ]:
import os
from pyspark.sql import SparkSession

# Add Iceberg + AWS dependencies for S3 access (without iceberg-aws to avoid constructor issues)
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.4.2,"
    "software.amazon.awssdk:s3:2.20.89,"
    "software.amazon.awssdk:auth:2.20.89 pyspark-shell"
)

# Create Spark session with Unity Catalog configuration
spark = SparkSession.builder \
    .appName("Unity Catalog Iceberg Client") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.type", "rest") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.uri", f"https://{WORKSPACE_URI}/api/2.1/unity-catalog/iceberg-rest") \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.warehouse", UC_CATALOG_NAME) \
    .config(f"spark.sql.catalog.{UC_CATALOG_NAME}.token", ACCESS_TOKEN) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
    .getOrCreate()

print("Spark session created successfully!")
print(f"Spark version: {spark.version}")
print(f"Configured catalog: {UC_CATALOG_NAME}")


:: loading settings :: url = jar:file:/Users/alexander.booth/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/alexander.booth/.ivy2/cache
The jars for the packages stored in: /Users/alexander.booth/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#auth added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3ecc4b0e-b938-4db4-857b-cbb411cf14dd;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.2 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.4.2 in central
	found software.amazon.awssdk#s3;2.20.89 in central
	found software.amazon.awssdk#aws-xml-protocol;2.20.89 in central
	found software.amazon.awssdk#aws-query-protocol;2.20.89 in central
	found software.amazon.awssdk#protocol-core;2.20.89 in central
	found software.amazon.awssdk#sdk-core;2.20.89 in central
	found software.amazon.awssdk#annotations;2.20.89 in central
	found so

Spark session created successfully!
Spark version: 3.5.1
Configured catalog: alexander_booth


In [ ]:
# Query the iceberg_test table
df = spark.sql("SELECT * FROM alexander_booth.default.iceberg_test")
print("Query executed successfully!")
print(f"Number of rows: {df.count()}")
print("\nFirst 10 rows:")
df.show()


Query executed successfully!
Number of rows: 10

First 20 rows:


+---+--------------+-------------------+---------+
| id|          name|         created_at|is_active|
+---+--------------+-------------------+---------+
|  1| Alice Johnson|2024-07-01 05:30:00|     true|
|  2|  Bob Martinez|2024-07-02 09:45:00|    false|
|  5|    Evan Patel|2024-07-05 07:00:00|    false|
|  6| Fatima Hassan|2024-07-06 03:05:00|     true|
|  8|   Hana Suzuki|2024-07-08 06:25:00|     true|
|  3|Charlie Nguyen|2024-07-03 04:15:00|     true|
|  4|       Dana Li|2024-07-04 11:20:00|     true|
|  7|    George Kim|2024-07-07 13:40:00|    false|
|  9|   Ivan Petrov|2024-07-09 08:55:00|     true|
| 10|  Julia Garcia|2024-07-10 12:30:00|    false|
+---+--------------+-------------------+---------+



In [3]:
# Delete row with id = 5
spark.sql("DELETE FROM alexander_booth.default.iceberg_test WHERE id = 5")
print("Row with id = 5 deleted successfully!")

# Re-query the table
df = spark.sql("SELECT * FROM alexander_booth.default.iceberg_test")
print("Query executed successfully!")
print(f"Number of rows after deletion: {df.count()}")
print("\nFirst 10 rows:")
df.show()

Row with id = 5 deleted successfully!
Query executed successfully!
Number of rows after deletion: 9

First 10 rows:
+---+--------------+-------------------+---------+
| id|          name|         created_at|is_active|
+---+--------------+-------------------+---------+
|  1| Alice Johnson|2024-07-01 05:30:00|     true|
|  2|  Bob Martinez|2024-07-02 09:45:00|    false|
|  6| Fatima Hassan|2024-07-06 03:05:00|     true|
|  8|   Hana Suzuki|2024-07-08 06:25:00|     true|
|  3|Charlie Nguyen|2024-07-03 04:15:00|     true|
|  4|       Dana Li|2024-07-04 11:20:00|     true|
|  7|    George Kim|2024-07-07 13:40:00|    false|
|  9|   Ivan Petrov|2024-07-09 08:55:00|     true|
| 10|  Julia Garcia|2024-07-10 12:30:00|    false|
+---+--------------+-------------------+---------+



In [ ]:
# Stop the Spark session when done
spark.stop()
print("Spark session stopped successfully!")
